# Loan Default Prediction 

### ingestion

loading the raw data and verifying.

In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("yasserh/loan-default-dataset")
file_path = os.path.join(path, "Loan_Default.csv")
data = pd.read_csv(file_path)

/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Validating the data

In [2]:
data.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [3]:
data.shape

(148670, 34)

In [4]:
data.columns

Index(['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type',
       'loan_purpose', 'Credit_Worthiness', 'open_credit',
       'business_or_commercial', 'loan_amount', 'rate_of_interest',
       'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization',
       'interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
       'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type',
       'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status', 'dtir1'],
      dtype='str')

In [5]:
X = data.drop(columns=['Status','Gender','ID','year'])   
Y = data['Status']

### pre-processing
Handles nulls and encoding

After doing the explanatory data analysis strategy for imputation of NaN values for numerical features which exhibits the skewness by median and for those numerical features, which does not exhibits skewness by median(since both mean and median will be the same, but to ensure consistency throughout the data media will be used for imputation)

For categorical features, the strategy for imputation (after doing the data analysis of the features with other features through cross tabulation and not finding a supporting relationship between them) by mode.


In [6]:
num_cols = [
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "property_value",
    "income",
    "LTV",
    "dtir1"
]

cat_cols = [
    "loan_limit",
    "approv_in_adv",
    "submission_of_application",
    "age",
    "loan_purpose",
    "Neg_ammortization"
]

# Numerical
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# Categorical
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])



One-Hot Encoding categorical features.

In [ ]:
Cat_cols = [col for col in X.columns if X[col].dtype in ['str', 'category']]
#One-Hot Encoding 

X = pd.get_dummies(data = X,
                         prefix = Cat_cols,
                         columns = Cat_cols)

### Feature Selection



Pearson Correlation Coefficients for Numerical Features

In [13]:
from scipy import stats
Num_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

num_df = X[Num_cols]
pearson_dic = {i: stats.pearsonr(num_df[i], data['Status']) for i in num_df.columns}
pd.DataFrame(pearson_dic, index = ['Pearson Statistic', 'p-value']).T.sort_values(by='Pearson Statistic', ascending = False)

,Pearson Statistic,p-value
dtir1,0.082432,1.929235e-222
LTV,0.042656,7.788582e-61
loan_amount,-0.036825,8.690628e-46
income,-0.060618,4.882970e-121
property_value,-0.080905,2.522136e-214


Chi-Square test of Categorical Variables¶


In [9]:

def chi_square_test(cat_var, target_var):

    contingency_table = pd.crosstab(data[cat_var], data[target_var])
    chi2, p, _, _ = stats.chi2_contingency(contingency_table)
    return chi2, p, contingency_table


chi_square_results = []


for col in Cat_cols:
    chi2, p, contingency_table = chi_square_test(col, 'Status')
    chi_square_results.append({
        'Variable': col,
        'Chi2 Statistic': chi2,
        'P-Value': p,
        'Contingency Table': contingency_table
    })


chi_square_results_df = pd.DataFrame(chi_square_results)


chi_square_results_df.sort_values(by='Chi2 Statistic', ascending = False)

,Variable,Chi2 Statistic,P-Value,Contingency Table
14,credit_type,52135.280705,0.000000e+00,Status 0 1 credit_type ...
9,lump_sum_payment,5237.827442,0.000000e+00,Status 0 1 lump_sum_payme...
7,Neg_ammortization,3610.208104,0.000000e+00,Status 0 1 Neg_ammortiza...
15,co-applicant_credit_type,3092.392571,0.000000e+00,Status 0 1 co-appl...
17,submission_of_application,2171.709755,0.000000e+00,Status 0 1 submis...
2,loan_type,1309.958143,3.517253e-285,Status 0 1 loan_type ...
6,business_or_commercial,1272.807998,9.172191e-279,Status 0 1 business_...
0,loan_limit,427.398487,5.985647e-95,Status 0 1 loan_limit ...
18,Region,380.456330,3.786057e-82,Status 0 1 Region ...
16,age,374.501644,8.442234e-78,Status 0 1 age 25-34 ...


Mutual Information (for Categorical and Numerical Features)


In [14]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X, Y, discrete_features = 'auto')
mi_scores_df = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("Mutual Information Scores:")
print(mi_scores_df.to_string())

Mutual Information Scores:
LTV                             0.176476
credit_type_EQUI                0.164159
property_value                  0.139249
dtir1                           0.087424
lump_sum_payment_not_lpsm       0.034060
Neg_ammortization_not_neg       0.027180
co-applicant_credit_type_CIB    0.026821
interest_only_not_int           0.026111
co-applicant_credit_type_EXP    0.025603
Credit_Worthiness_l1            0.025464
income                          0.024812
business_or_commercial_nob/c    0.022909
loan_type_type1                 0.022835
credit_type_CIB                 0.019330
approv_in_adv_nopre             0.018466
Region_North                    0.017657
loan_limit_cf                   0.014943
credit_type_CRIF                0.014447
credit_type_EXP                 0.014443
lump_sum_payment_lpsm           0.014039
occupancy_type_pr               0.013726
Region_south                    0.013488
Neg_ammortization_neg_amm       0.010541
loan_purpose_p3               

Final selected feature for model training

In [8]:
num_features = ['income', 'loan_amount', 'dtir1', 'LTV', 'property_value']


cat_features = ['loan_limit', 'approv_in_adv', 'loan_type', 'loan_purpose', 
               'Credit_Worthiness', 'business_or_commercial', 'Neg_ammortization',
               'interest_only', 'lump_sum_payment', 'occupancy_type', 'credit_type',
               'co-applicant_credit_type', 'Region']

features = num_features + cat_features

In [18]:
from sklearn.model_selection import train_test_split

X_t = data[features].copy()
Y_t = data['Status']

num_cols = [
    "property_value",
    "income",
    "LTV",
    "dtir1"
]

cat_cols = [
    "loan_limit",
    "approv_in_adv",
    "loan_purpose",
    "Neg_ammortization"
]

# Numerical
for col in num_cols:
    X_t[col] = X_t[col].fillna(X_t[col].median())

# Categorical
for col in cat_cols:
    X_t[col] = X_t[col].fillna(X_t[col].mode()[0])


X_t = pd.get_dummies(data = X_t,
                         prefix = cat_features,
                         columns = cat_features)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X_t, Y_t, train_size = 0.8, random_state = 0)

### Model Building

Building a random forest model

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, min_samples_split = 2, max_depth = 10, random_state=0)

rf_pipe = Pipeline(steps=[
    ('classifier', rf)
])

rf_pipe.fit(X_train, y_train)

# Prediction
y_pred = rf_pipe.predict(X_test)

# Evaluation
print(f"Model Accuracy: {rf_pipe.score(X_test, y_test)}\n")

print(f"Confusion Matrix: \n {confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report : \n {classification_report(y_test, y_pred)}\n")


print(f"ROC-AUC Score: \n {roc_auc_score(y_test, rf_pipe.predict_proba(X_test)[:, 1])}")


Model Accuracy: 0.8722001748839712

Confusion Matrix: 
 [[21008  1318]
 [ 2482  4926]]

Classification Report : 
               precision    recall  f1-score   support

           0       0.89      0.94      0.92     22326
           1       0.79      0.66      0.72      7408

    accuracy                           0.87     29734
   macro avg       0.84      0.80      0.82     29734
weighted avg       0.87      0.87      0.87     29734


ROC-AUC Score: 
 0.876041147896021


from sklearn.model_selection import RandomizedSearchCV


params = {
    'classifier__n_estimators': [100, 200, 300],   
    'classifier__max_depth': [5, 10, None],          
    'classifier__min_samples_split': [2, 5, 10],          
    'classifier__min_samples_leaf': [1, 2, 4],        
    'classifier__max_features': ['sqrt', 'log2']  
}


random_search = RandomizedSearchCV(rf_pipe, param_distributions=params, 
                                   n_iter=50, cv=5, scoring='roc_auc', verbose=2, n_jobs=-1, random_state=0)


random_search.fit(X_train, y_train)

# Get the best parameters and model
best_model_rf = random_search.best_estimator_
print(random_search.best_params_)

# Evaluating the best model
y_pred_best = best_model_rf.predict(X_test)
print(f"Best Model Accuracy: {best_model_rf.score(X_test, y_test)}\n")

print(f"Classification Report : \n{classification_report(y_test, y_pred_best)}\n")

roc_auc = roc_auc_score(y_test, best_model_rf.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {roc_auc}")


XGBoost model

In [20]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_pipeline = Pipeline(steps=[
    ('classifier', xgb_model)
])

xgb_pipeline.fit(X_train, y_train)


# Prediction
y_pred = xgb_pipeline.predict(X_test)

# Evaluation
print(f"Model Accuracy: {xgb_pipeline.score(X_test, y_test)}\n")

print(f"Confusion Matrix: \n {confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report : \n {classification_report(y_test, y_pred)}\n")


print(f"ROC-AUC Score: \n {roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:, 1])}")

/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [03:16:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model Accuracy: 0.8937243559561445

Confusion Matrix: 
 [[21926   400]
 [ 2760  4648]]

Classification Report : 
               precision    recall  f1-score   support

           0       0.89      0.98      0.93     22326
           1       0.92      0.63      0.75      7408

    accuracy                           0.89     29734
   macro avg       0.90      0.80      0.84     29734
weighted avg       0.90      0.89      0.89     29734


ROC-AUC Score: 
 0.8873066424505981


In [21]:
from sklearn.model_selection import RandomizedSearchCV

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_pipeline = Pipeline(steps=[
    ('classifier', xgb_model)
])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1],
    'classifier__max_depth': [3, 7, 10]
}
random_search = RandomizedSearchCV(xgb_pipeline, param_grid, cv=5, scoring='roc_auc',
                                  n_jobs=-1, random_state=0)

random_search.fit(X_train, y_train)


# Get the best parameters and model
best_model_xgb = random_search.best_estimator_
print(random_search.best_params_)

# Evaluating the best model
y_pred_best = best_model_xgb.predict(X_test)
print(f"Best Model Accuracy: {best_model_xgb.score(X_test, y_test)}\n")

print(f"Classification Report : \n{classification_report(y_test, y_pred_best)}\n")

roc_auc = roc_auc_score(y_test, best_model_xgb.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {roc_auc}")


/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [03:16:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [03:16:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [03:16:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [03:16:38] WARNING:

{'classifier__n_estimators': 200, 'classifier__max_depth': 7, 'classifier__learning_rate': 0.1}
Best Model Accuracy: 0.8953723010694827

Classification Report : 
              precision    recall  f1-score   support

           0       0.89      0.98      0.93     22326
           1       0.93      0.63      0.75      7408

    accuracy                           0.90     29734
   macro avg       0.91      0.81      0.84     29734
weighted avg       0.90      0.90      0.89     29734


ROC-AUC Score: 0.8900907418134848
